In [1]:
import pandas as pd


In [2]:
df= pd.read_csv("data/datos_para_limpiar.csv")

In [3]:
print
df.head(5)


,ID,Nombre,Edad,Salario,Ciudad,Genero,Nivel,Fecha,Ingresos,Gastos,Codigo,Notas
0,1,Ana,NaN,3.955413e+03,bogota,Masculino,Bajo,2018-12-29,4030.026008,1213.936751,ABC12345,Pendiente evaluación
1,2,Maria,NaN,3.054891e+03,NaN,Femenino,alto,2011-07-16,7509.810584,6038.896355,NaN,Buen empleado
2,3,Carlos,NaN,2.928345e+08,bogota,Femenino,Bajo,2022-04-07,9106.362328,6705.724812,NaN,NaN
3,4,Maria,11.0,4.718402e+03,NaN,NaN,medio,2022-10-11,NaN,NaN,XYZ98765,NaN
4,5,Laura,94.0,7.477876e+07,cali,masculino,alto,2017-07-11,3102.167244,7570.107277,NaN,Pendiente evaluación


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 12 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   ID        100 non-null    int64  
 1   Nombre    100 non-null    str    
 2   Edad      50 non-null     float64
 3   Salario   65 non-null     float64
 4   Ciudad    82 non-null     str    
 5   Genero    73 non-null     str    
 6   Nivel     73 non-null     str    
 7   Fecha     100 non-null    str    
 8   Ingresos  48 non-null     float64
 9   Gastos    53 non-null     float64
 10  Codigo    69 non-null     str    
 11  Notas     67 non-null     str    
dtypes: float64(4), int64(1), str(7)
memory usage: 9.5 KB


In [5]:
print(df.isnull().sum())

ID           0
Nombre       0
Edad        50
Salario     35
Ciudad      18
Genero      27
Nivel       27
Fecha        0
Ingresos    52
Gastos      47
Codigo      31
Notas       33
dtype: int64


In [7]:
df = df.drop(columns=["ID", "Notas"])

In [ ]:

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.normalize('NFKD')
    .str.encode('ascii', errors='ignore')
    .str.decode('utf-8')
    .str.replace(' ', '_')
)

#imprime si quedo correcto o no 
print("Nuevos nombres de columnas:", df.columns.tolist())

Nuevos nombres de columnas: ['nombre', 'edad', 'salario', 'ciudad', 'genero', 'nivel', 'fecha', 'ingresos', 'gastos', 'codigo']


In [ ]:

df['ciudad'] = (
    df['ciudad']
    .str.strip()
    .str.lower()
    .str.normalize('NFKD')
    .str.encode('ascii', errors='ignore')
    .str.decode('utf-8')
)

df['ciudad'] = df['ciudad'].replace({
    'bogota': 'bogota',
    'medellin': 'medellin',
    'cali': 'cali'
})

print("Ciudades unificadas:", df['ciudad'].unique())

Ciudades unificadas: <StringArray>
['bogota', nan, 'cali', 'medellin']
Length: 4, dtype: str


In [ ]:

edad_validas = df[(df['edad'] >= 0) & (df['edad'] <= 120)]['edad']
media_edad = edad_validas.mean()
df['edad'] = df['edad'].fillna(media_edad)

mediana_salario = df['salario'].median()
df['salario'] = df['salario'].fillna(mediana_salario)

moda_ciudad = df['ciudad'].mode()[0]
df['ciudad'] = df['ciudad'].fillna(moda_ciudad)

df['genero'] = df['genero'].str.strip().str.lower()
moda_genero = df['genero'].mode()[0]
df['genero'] = df['genero'].fillna(moda_genero)

print("Nuevos conteos de nulos en estas columnas:")
print(df[['edad', 'salario', 'ciudad', 'genero']].isnull().sum())

Nuevos conteos de nulos en estas columnas:
edad       0
salario    0
ciudad     0
genero     0
dtype: int64


In [ ]:
df['edad'] = pd.to_numeric(df['edad'], errors='coerce')
df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')

print("Tipos de datos actualizados:")
print(df[['edad', 'fecha']].dtypes)

Tipos de datos actualizados:
edad            float64
fecha    datetime64[us]
dtype: object


In [ ]:
df = df[(df['edad'] >= 0) & (df['edad'] <= 120)]

df = df[df['salario'] <= 1_000_000_000]

print(f"Total de registros válidos después del filtrado: {len(df)}")

Total de registros válidos después del filtrado: 93


In [ ]:
df['nivel'] = df['nivel'].astype(str).str.strip().str.lower()

df['nivel'] = df['nivel'].replace(['n/a', 'nan', 'none', ''], pd.NA)

orden_nivel = {
    'bajo': 1, 
    'medio': 2, 
    'alto': 3
}

df['nivel'] = df['nivel'].map(orden_nivel)

print(df['nivel'].value_counts(dropna=False))


nivel
1.0    28
2.0    26
NaN    25
3.0    14
Name: count, dtype: int64


In [ ]:
df = pd.get_dummies(df, columns=['genero', 'ciudad'], drop_first=True)

print("Columnas después del get_dummies:")
print(df.columns.tolist())


Columnas después del get_dummies:
['nombre', 'edad', 'salario', 'nivel', 'fecha', 'ingresos', 'gastos', 'codigo', 'genero_masculino', 'genero_otro', 'ciudad_cali', 'ciudad_medellin']


In [ ]:
from sklearn.preprocessing import MinMaxScaler

df['ingresos'] = df['ingresos'].fillna(df['ingresos'].median())
df['gastos'] = df['gastos'].fillna(df['gastos'].median())

# 10. 
scaler = MinMaxScaler()
df[['edad', 'ingresos']] = scaler.fit_transform(df[['edad', 'ingresos']])

print("¡Escalado aplicado con éxito!")

¡Escalado aplicado con éxito!


In [ ]:
df['utilidad'] = df['ingresos'] - df['gastos']

df.to_csv('datos_limpios.csv', index=False)

print("¡Proceso de limpieza completado con éxito!")
print("Se ha generado el archivo 'datos_limpios.csv' en tu directorio de trabajo.")

# Vista previa final del dataset limpio
display(df.head())

¡Proceso de limpieza completado con éxito!
Se ha generado el archivo 'datos_limpios.csv' en tu directorio de trabajo.


,nombre,edad,salario,nivel,fecha,ingresos,gastos,codigo,genero_masculino,genero_otro,ciudad_cali,ciudad_medellin,utilidad
0,Ana,58.790698,3.955413e+03,1.0,2018-12-29,4030.026008,1213.936751,ABC12345,True,False,False,False,2816.089257
1,Maria,58.790698,3.054891e+03,3.0,2011-07-16,7509.810584,6038.896355,NaN,False,False,False,False,1470.914229
2,Carlos,58.790698,2.928345e+08,1.0,2022-04-07,9106.362328,6705.724812,NaN,False,False,False,False,2400.637516
3,Maria,11.000000,4.718402e+03,2.0,2022-10-11,5794.942635,3898.626186,XYZ98765,False,False,False,False,1896.316449
4,Laura,94.000000,7.477876e+07,3.0,2017-07-11,3102.167244,7570.107277,NaN,True,False,True,False,-4467.940033
